# 🔍 AmbitionBox Job Scraper v3

**How it works:**
1. Load company listing page → scroll to load all `div.jobInfoCard` cards
2. From each card extract: **Title, Experience, Salary, Location, Detail URL, Apply Link**
   - Apply Link is built directly from the `?rid=naukri_XXXXXX` in the card URL — no clicking needed
3. Visit each detail page **only** for the **Job Description**
4. Save checkpoint CSV after each company

```bash
pip install selenium beautifulsoup4 lxml pandas webdriver-manager
```

In [16]:
import pandas as pd
import time
import random
import re
import os
from bs4 import BeautifulSoup as bs
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

try:
    from webdriver_manager.chrome import ChromeDriverManager
    USE_WDM = True
except ImportError:
    USE_WDM = False

print("✅ Imports OK")

✅ Imports OK


## ⚙️ Configuration

In [17]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
INPUT_CSV   = "company_data.csv"   # Input CSV with company names
OUTPUT_CSV  = "jobs_output.csv"    # Output CSV
COMPANY_COL = "Company Name"       # Column name in input CSV

# ── TIMING ─────────────────────────────────────────────────────────────────────
PAGE_LOAD_WAIT   = 15    # Max wait for elements (seconds)
SCROLL_PAUSE     = 2.5   # Pause between each infinite-scroll step
SCROLL_MAX_STALL = 3     # Stop scrolling after N stalls with no new cards
DETAIL_DELAY_MIN = 1.5   # Min sleep between detail page visits
DETAIL_DELAY_MAX = 3.0   # Max sleep between detail page visits
COMPANY_DELAY    = 4.0   # Sleep between companies

# ── RESUME ─────────────────────────────────────────────────────────────────────
RESUME_FROM_CHECKPOINT = True  # Skip companies already in OUTPUT_CSV

print("✅ Config ready")

✅ Config ready


In [18]:
# ── CHROME DRIVER ──────────────────────────────────────────────────────────────
def make_driver() -> webdriver.Chrome:
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("--window-size=1400,900")
    opts.add_experimental_option("excludeSwitches", ["enable-automation"])
    opts.add_experimental_option("useAutomationExtension", False)
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/122.0.0.0 Safari/537.36"
    )
    if USE_WDM:
        driver = webdriver.Chrome(
            service=Service(ChromeDriverManager().install()), options=opts
        )
    else:
        driver = webdriver.Chrome(options=opts)

    driver.execute_cdp_cmd(
        "Page.addScriptToEvaluateOnNewDocument",
        {"source": "Object.defineProperty(navigator,'webdriver',{get:()=>undefined})"},
    )
    return driver


def polite_sleep(mn=DETAIL_DELAY_MIN, mx=DETAIL_DELAY_MAX):
    time.sleep(random.uniform(mn, mx))


def company_to_slug(name: str) -> str:
    """'Tata Consultancy Services' → 'tata-consultancy-services'"""
    name = name.strip().lower()
    name = re.sub(r'[^a-z0-9\s-]', '', name)
    name = re.sub(r'[\s_]+', '-', name)
    return re.sub(r'-+', '-', name).strip('-')


def rid_to_apply_link(rid_value: str) -> str | None:
    """
    Converts rid parameter to Naukri apply URL.

    rid_value examples:
      'naukri_280326016167'  → Naukri ID = 280326016167
      'hirist_1621273'       → different job board, return None for now

    Naukri apply URL pattern (confirmed from browser network trace):
      https://www.naukri.com/job-listings-{ID}?utm_source=ambitionbox
                                                &utm_medium=ab_job_listing
                                                &utm_campaign=ambitionbox_jobs
    """
    if not rid_value:
        return None
    match = re.match(r'naukri_(\d+)', rid_value, re.IGNORECASE)
    if match:
        naukri_id = match.group(1)
        return (
            f"https://www.naukri.com/job-listings-{naukri_id}"
            "?utm_source=ambitionbox"
            "&utm_medium=ab_job_listing"
            "&utm_campaign=ambitionbox_jobs"
        )
    # Handle other job boards if present
    return None


print("✅ Helpers defined")

✅ Helpers defined


In [19]:
# ── STEP 1: LISTING PAGE → collect all job cards ───────────────────────────────
#
# Confirmed card structure:
#   <div class="jobInfoCard" itemprop="itemListElement">
#     <meta itemprop="url"  content="/jobs/tcs-jobs-cmp?rid=naukri_280326016217">
#     <meta itemprop="name" content="SAP Data Sphere">
#     <div class="job-basic-info">
#       <div class="entity" title="5-10 years">   ← experience
#       <div class="entity" title="₹ 2.5L/yr..."> ← salary
#       <div class="entity loc" title="Kolkata">   ← location (has 'loc' class)

def scroll_to_load_all(driver: webdriver.Chrome) -> None:
    """Scroll until no new job cards appear for SCROLL_MAX_STALL consecutive scrolls."""
    last_count  = 0
    stall_count = 0
    while stall_count < SCROLL_MAX_STALL:
        current = len(driver.find_elements(By.CSS_SELECTOR, "div.jobInfoCard"))
        if current == last_count:
            stall_count += 1
        else:
            stall_count = 0
            print(f"      ↳ {current} cards loaded...")
        last_count = current
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(SCROLL_PAUSE)
    print(f"      ✅ Scroll done — {last_count} total cards.")


def collect_listing_cards(driver: webdriver.Chrome, company_name: str) -> list[dict]:
    slug = company_to_slug(company_name)
    url  = f"https://www.ambitionbox.com/jobs/{slug}-jobs-cmp"
    print(f"   🌐 {url}")

    try:
        driver.get(url)
        WebDriverWait(driver, PAGE_LOAD_WAIT).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "div.jobInfoCard"))
        )
    except TimeoutException:
        print(f"   ⚠️  No job cards found (slug: {slug}) — skipping.")
        return []

    scroll_to_load_all(driver)
    soup  = bs(driver.page_source, "lxml")
    cards = soup.select("div.jobInfoCard")
    jobs  = []

    for card in cards:
        # ── Detail URL & rid ──────────────────────────────────────────────
        meta_url = card.find("meta", itemprop="url")
        if not meta_url:
            continue
        raw_url = meta_url.get("content", "")
        detail_url = (
            raw_url if raw_url.startswith("http")
            else "https://www.ambitionbox.com" + raw_url
        )

        # Extract rid value (e.g. 'naukri_280326016217')
        rid_match = re.search(r'rid=([^&]+)', raw_url)
        rid_value = rid_match.group(1) if rid_match else None

        # ── Apply Link — built directly from rid, no clicking ─────────────
        apply_link = rid_to_apply_link(rid_value)

        # ── Job Title ─────────────────────────────────────────────────────
        meta_name = card.find("meta", itemprop="name")
        title = meta_name.get("content", "").strip() if meta_name else ""
        if not title:
            a = card.select_one("h2 a.title")
            title = a.get_text(strip=True) if a else "N/A"

        # ── Experience, Salary, Location from entity title= attrs ─────────
        experience = "N/A"
        salary     = "N/A"
        location   = "N/A"

        for ent in card.select("div.job-basic-info div.entity"):
            t      = (ent.get("title") or "").strip()
            cls    = " ".join(ent.get("class", []))
            if not t:
                continue
            if re.search(r'\d+.*[Yy]r', t):
                experience = t
            elif "₹" in t or re.search(r'[Ll][Pp][Aa]|l/yr', t, re.I):
                salary = t
            elif "loc" in cls:
                location = t

        jobs.append({
            "Company Name":    company_name,
            "Job Title":       title,
            "Experience":      experience,
            "Salary":          salary,
            "Location":        location,
            "Apply Link":      apply_link,
            "Detail URL":      detail_url,   # used to fetch description; dropped in output
            "Job Description": None,
        })

    print(f"   📋 {len(jobs)} jobs parsed from listing page.")
    return jobs


print("✅ Listing scraper defined")

✅ Listing scraper defined


In [20]:
# ── STEP 2: DETAIL PAGE → Job Description only ─────────────────────────────────
#
# Confirmed selector:
#   div[data-testid="jd-description-html"]  → full job description HTML

def get_job_description(driver: webdriver.Chrome, detail_url: str) -> str:
    """Visit the detail page and return plain-text job description."""
    try:
        driver.get(detail_url)
        WebDriverWait(driver, PAGE_LOAD_WAIT).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'h2[data-testid="jd-job-title"]')
            )
        )
        soup     = bs(driver.page_source, "lxml")
        desc_div = soup.select_one('div[data-testid="jd-description-html"]')
        if not desc_div:
            return "N/A"
        text = desc_div.get_text(separator=" ", strip=True)
        return re.sub(r'\s+', ' ', text).strip()
    except TimeoutException:
        return "N/A"
    except Exception as e:
        print(f"         ⚠️  {e}")
        return "N/A"


print("✅ Detail scraper defined")

✅ Detail scraper defined


In [21]:
# ── FULL COMPANY PIPELINE ───────────────────────────────────────────────────────
def scrape_company(driver: webdriver.Chrome, company_name: str) -> list[dict]:
    # Step 1: listing page — gets everything except description
    jobs = collect_listing_cards(driver, company_name)
    if not jobs:
        return []

    # Step 2: visit each detail page only for job description
    print(f"   📝 Fetching job descriptions ({len(jobs)} pages)...")
    for i, job in enumerate(jobs):
        desc = get_job_description(driver, job["Detail URL"])
        jobs[i]["Job Description"] = desc
        apply_status = "✓" if job["Apply Link"] else "✗"
        desc_status  = "✓" if desc != "N/A" else "✗"
        print(
            f"      [{i+1:3d}/{len(jobs)}] "
            f"Apply:{apply_status} Desc:{desc_status} "
            f"| '{job['Job Title']}'"
        )
        polite_sleep()

    return jobs


print("✅ Orchestrator defined")

✅ Orchestrator defined


In [22]:
# ── LOAD COMPANIES ──────────────────────────────────────────────────────────────
df_companies  = pd.read_csv(INPUT_CSV)
company_names = df_companies[COMPANY_COL].dropna().str.strip().unique().tolist()
print(f"✅ {len(company_names)} companies loaded.")
print("First 5:", company_names[:5])

already_scraped = set()
if RESUME_FROM_CHECKPOINT and os.path.exists(OUTPUT_CSV):
    existing = pd.read_csv(OUTPUT_CSV)
    already_scraped = set(existing["Company Name"].dropna().unique())
    print(f"♻️  Skipping {len(already_scraped)} already-scraped companies.")

to_scrape = [c for c in company_names if c not in already_scraped]
print(f"🚀 {len(to_scrape)} companies to scrape.")

✅ 9497 companies loaded.
First 5: ['TCS', 'Accenture', 'Wipro', 'Cognizant', 'Capgemini']
🚀 9497 companies to scrape.


In [27]:
# ── MAIN LOOP ───────────────────────────────────────────────────────────────────
driver = make_driver()
all_results = []

start = 41   # scraper serial to start from (1-based, inclusive)    # History: 1
end   = 42  # scraper serial to end at   (1-based, inclusive)    # History: 40

try:
    for idx, company in enumerate(to_scrape[start - 1 : end], start=start):
        print(f"\n{'='*65}")
        print(f"[{idx}/{end}] {company}")
        print(f"{'='*65}")

        try:
            jobs = scrape_company(driver, company)
            all_results.extend(jobs)

            if jobs:
                df_chunk = pd.DataFrame(jobs).drop(
                    columns=["Detail URL"], errors="ignore"
                )
                write_header = not os.path.exists(OUTPUT_CSV)
                df_chunk.to_csv(
                    OUTPUT_CSV, mode="a", header=write_header, index=False
                )
                print(f"   💾 {len(jobs)} rows saved → {OUTPUT_CSV}")
            else:
                print(f"   ⚠️  No jobs found.")

        except Exception as e:
            print(f"   ❌ Fatal error for '{company}': {e}")

        if idx < end:
            print(f"   😴 Sleeping {COMPANY_DELAY}s...")
            time.sleep(COMPANY_DELAY)

finally:
    driver.quit()
    print("\n🛑 Driver closed.")

print(f"\n✅ Done! Jobs collected this run: {len(all_results)}")


[41/42] HDFC Life
   🌐 https://www.ambitionbox.com/jobs/hdfc-life-jobs-cmp
      ↳ 20 cards loaded...
      ✅ Scroll done — 20 total cards.
   📋 20 jobs parsed from listing page.
   📝 Fetching job descriptions (20 pages)...
      [  1/20] Apply:✓ Desc:✓ | 'Branch Manager - Agency ( Kolkata )'
      [  2/20] Apply:✓ Desc:✓ | 'Senior Associate-Branch Operation'
      [  3/20] Apply:✓ Desc:✓ | 'Senior Associate-Branch Operation'
      [  4/20] Apply:✓ Desc:✓ | 'Associate Relationship Manager'
      [  5/20] Apply:✓ Desc:✓ | 'Senior Associate Branch Operations'
      [  6/20] Apply:✓ Desc:✓ | 'Senior Associate-Branch Operation'
      [  7/20] Apply:✓ Desc:✓ | 'Senior Associate-Branch Operation'
      [  8/20] Apply:✓ Desc:✓ | 'Senior Manager - Lead Integration Developer'
      [  9/20] Apply:✓ Desc:✓ | 'Corporate Agency Manager'
      [ 10/20] Apply:✓ Desc:✓ | 'SDM/ CAM - Banca Others channel - Surendranagar'
      [ 11/20] Apply:✓ Desc:✓ | 'SDM/ CAM - Banca Others channel - Valsad'
     

In [28]:
# ── FINAL CLEAN & DEDUPLICATE ───────────────────────────────────────────────────
if os.path.exists(OUTPUT_CSV):
    df_final = pd.read_csv(OUTPUT_CSV)
    df_final = df_final.drop(columns=["Detail URL"], errors="ignore")
    df_final = df_final.drop_duplicates(
        subset=["Company Name", "Job Title", "Apply Link"]
    )
    df_final.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Final CSV saved: {OUTPUT_CSV}")
    print(f"   Rows : {len(df_final)}")
    print(f"   Cols : {list(df_final.columns)}")
    display(df_final.head(10))
else:
    print("⚠️  No output file found.")

✅ Final CSV saved: jobs_output.csv
   Rows : 772
   Cols : ['Company Name', 'Job Title', 'Salary', 'Experience', 'Location', 'Apply Link', 'Job Description']


,Company Name,Job Title,Salary,Experience,Location,Apply Link,Job Description
0,TCS,SAP Data Sphere,₹ 2.5L/yr - 28L/yr,NaN,Kolkata,https://www.naukri.com/job-listings-2803260162...,Greetings from TCS ! *Face to Face*-Walk-In In...
1,TCS,ServiceNow ITOM/SPM/CSM/ITAM,₹ 2.5L/yr - 28L/yr,NaN,Kolkata,https://www.naukri.com/job-listings-2803260161...,Greetings from TCS ! *Face to Face*-Walk-In In...
2,TCS,SAP ISU Billing,NaN,NaN,Kolkata,https://www.naukri.com/job-listings-2803260161...,Greetings from TCS ! *Face to Face*-Walk-In In...
3,TCS,Oracle DBA Analyst,₹ 12L/yr - 18L/yr,NaN,"Hyderabad, Bengaluru, Delhi/Ncr",https://www.naukri.com/job-listings-2803260161...,"Key Responsibilities: Install, configure, upgr..."
4,TCS,SAP MM,NaN,NaN,Kolkata,https://www.naukri.com/job-listings-2803260161...,Greetings from TCS ! *Face to Face*-Walk-In In...
5,TCS,SAP BW HANA Consultant,₹ 12L/yr - 18L/yr,NaN,"Hyderabad, Chennai, Bengaluru",https://www.naukri.com/job-listings-2803260161...,Key Responsibilities: Design and develop BW on...
6,TCS,Oracle HCM,NaN,NaN,Kolkata,https://www.naukri.com/job-listings-2803260161...,Greetings from TCS ! *Face to Face*-Walk-In In...
7,TCS,Workday Integration Consultant,NaN,NaN,Kolkata,https://www.naukri.com/job-listings-2803260160...,Greetings from TCS ! *Face to Face*-Walk-In In...
8,TCS,Oracle Fusion Financials,NaN,NaN,Kolkata,https://www.naukri.com/job-listings-2803260160...,Greetings from TCS ! *Face to Face*-Walk-In In...
9,TCS,SAP Security & Authorizations,₹ 10L/yr - 16L/yr,NaN,"Hyderabad, Pune, Bengaluru",https://www.naukri.com/job-listings-2803260160...,Key Responsibilities: Design and maintain SAP ...


In [29]:
# ── SUMMARY STATS ───────────────────────────────────────────────────────────────
if os.path.exists(OUTPUT_CSV):
    df = pd.read_csv(OUTPUT_CSV)

    print("📊 Jobs per company (top 20):")
    print(df["Company Name"].value_counts().head(20).to_string())

    total = len(df)
    found = df["Apply Link"].notna().sum()
    print(f"\n🔗 Apply Link found : {found}/{total} ({100*found//max(total,1)}%)")

    no_desc = (df["Job Description"] == "N/A").sum()
    print(f"📝 Description N/A  : {no_desc}/{total}")

    print("\n📍 Top 10 Locations:")
    print(df["Location"].value_counts().head(10).to_string())

📊 Jobs per company (top 20):
Company Name
TCS                        20
Accenture                  20
Wipro                      20
Cognizant                  20
Capgemini                  20
HDFC Bank                  20
Infosys                    20
HCLTech                    20
Tech Mahindra              20
Genpact                    20
TP                         20
Axis Bank                  20
Amazon                     20
Reliance Retail            20
iEnergizer                 20
LTIMindtree                20
IBM                        20
HDB Financial Services     20
Larsen & Toubro Limited    20
Deloitte                   20

🔗 Apply Link found : 761/772 (98%)
📝 Description N/A  : 0/772

📍 Top 10 Locations:
Location
Bengaluru                     143
Chennai                        58
Pune                           56
Mumbai                         54
Hyderabad                      39
Noida                          39
Kolkata                        22
Gurugram                   